# Llama Surgery (Continual Sparsification) Starter

This notebook demonstrates the setup for Llama Surgery as described in the V3 roadmap.
We define a single transformer layer using `SurgicalLlamaAttention` and `MultiPrimeTopologyRouter`, and demonstrate the surgical forward pass and the auxiliary loss computation.

In [ ]:
import sys
import os
import torch

# Add src to path
sys.path.insert(0, os.path.abspath('../../'))

from src.ultrametric_v3.surgery import MultiPrimeTopologyRouter, SurgicalLlamaAttention, SurgeryLossRamp

# Hyperparameters
batch_size = 2
seq_len = 128
embed_dim = 256
num_heads = 4
p = 2
tau = 1.0
step = 100


In [ ]:
# Initialize modules
router = MultiPrimeTopologyRouter(num_heads=num_heads, init_val=-5.0)
attention = SurgicalLlamaAttention(embed_dim=embed_dim, num_heads=num_heads, p=p)
loss_ramp = SurgeryLossRamp(lambda_init=0.0, lambda_max=1.0, ramp_steps=1000)

# Dummy input
x = torch.randn(batch_size, seq_len, embed_dim)

# 1. Routing
g_h = router(tau=tau)
print(f"Routing gates (g_h): {g_h.detach().numpy()}")

# 2. Attention Forward Pass
out = attention(x, g_h)
print(f"Attention output shape: {out.shape}")

# 3. Compute auxiliary loss
aux_loss = loss_ramp(g_h, step=step)
print(f"Auxiliary loss at step {step}: {aux_loss.item():.4f}")

# Simulating step 1000 (fully polarized)
g_h_polarized = router(tau=0.1)
aux_loss_polarized = loss_ramp(g_h_polarized, step=1000)
print(f"Routing gates (g_h) at tau=0.1: {g_h_polarized.detach().numpy()}")
print(f"Auxiliary loss at step 1000: {aux_loss_polarized.item():.4f}")
